# v1 Assists — Flat Rule Backtest: All Lines

No model. No features. For every player-game with a `player_assists` line, bet the under.  
Goal: find which line tier (if any) has structural market inefficiency.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
sys.path.insert(0, str(repo_root))
from src.nba_rebounds_modeling.duckdb_s3_creds import connect_duckdb_s3

SEASONS = ["2023-24", "2024-25", "2025-26"]
SEASON_DATE_RANGES = {
    "2023-24": ("2023-10-01", "2024-06-30"),
    "2024-25": ("2024-10-01", "2025-06-30"),
    "2025-26": ("2025-10-01", "2026-06-30"),
}
MARKET = "player_assists"
TARGET_COL = "AST"
MIN_MINUTES = 10

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds): return float("nan")
    if odds < 0: return (-odds) / ((-odds) + 100.0)
    return 100.0 / (odds + 100.0)

def american_profit(odds: float) -> float:
    if odds >= 0: return odds / 100.0
    return 100.0 / (-odds)

con = connect_duckdb_s3()
print("Connected to DuckDB S3")

In [ ]:
logs_frames = []
for season in SEASONS:
    query = f"""
        SELECT
            PLAYER_NAME,
            CAST(AST AS DOUBLE) AS AST,
            CAST(MIN AS DOUBLE) AS MIN,
            GAME_DATE
        FROM read_csv_auto('s3://nba-api-mt/player_game_logs/{season}/*.csv',
                           header=true, ignore_errors=true)
    """
    frame = con.execute(query).df()
    frame["season"] = season
    logs_frames.append(frame)

logs = pd.concat(logs_frames, ignore_index=True)
logs = logs[logs["MIN"] >= MIN_MINUTES].copy()
logs["GAME_DATE"] = pd.to_datetime(logs["GAME_DATE"], format="mixed").dt.date
logs["player_key"] = logs["PLAYER_NAME"].str.lower().str.strip()
print(f"Game logs loaded (MIN >= {MIN_MINUTES}): {len(logs):,} rows")
print(logs.groupby("season")["PLAYER_NAME"].count())
print(f"AST column present: {'AST' in logs.columns}")

# Load props (all lines — no line filter)
props_frames = []
for season in SEASONS:
    start_date, end_date = SEASON_DATE_RANGES[season]
    query = f"""
        SELECT
            player,
            CAST(prop_line AS DOUBLE) AS prop_line,
            CAST(over_odds AS DOUBLE) AS over_odds,
            CAST(under_odds AS DOUBLE) AS under_odds,
            game_time
        FROM read_csv_auto('s3://the-odds-api-mt/nba/historical_player_props/{season}/*.csv',
                           header=true, ignore_errors=true)
        WHERE market = '{MARKET}'
          AND game_time >= '{start_date}'
          AND game_time <= '{end_date}'
    """
    frame = con.execute(query).df()
    frame["season"] = season
    props_frames.append(frame)

props_raw = pd.concat(props_frames, ignore_index=True)
props_raw["game_time"] = pd.to_datetime(props_raw["game_time"], format="mixed")
props_raw["game_date"] = props_raw["game_time"].dt.date
props_raw["player_key"] = props_raw["player"].str.lower().str.strip()

props = (
    props_raw
    .groupby(["player_key", "game_date", "season"], as_index=False)
    .agg(
        player=("player", "first"),
        prop_line=("prop_line", "first"),
        over_odds=("over_odds", "median"),
        under_odds=("under_odds", "median"),
    )
)
print(f"\nProps loaded (consensus): {len(props):,} rows")
print(props.groupby("season")["player_key"].count())

df = pd.merge(props, logs, on=["player_key", "game_date", "season"], how="inner")
print(f"\nJoined: {len(df):,} rows")
print(df.groupby("season")["player_key"].count())

In [ ]:
print("=== Line distribution ===")
print(df["prop_line"].value_counts().sort_index())
print(f"\nMost common line: {df['prop_line'].mode()[0]}")
print(f"Total unique lines: {df['prop_line'].nunique()}")

df["line_tier"] = pd.cut(df["prop_line"],
    bins=[0, 2, 4, 6, 100],
    labels=["low (0.5-1.5)", "mid (2.5-3.5)", "high (4.5-5.5)", "star (6.5+)"])

print("\n=== Line tier distribution ===")
tier_counts = df["line_tier"].value_counts()
print(tier_counts)
print("\nPct:")
print((tier_counts / len(df) * 100).round(1))

In [ ]:
df["vig"] = (df["over_odds"].apply(american_to_implied_prob)
           + df["under_odds"].apply(american_to_implied_prob))

df_clean = df[
    (df["vig"] >= 1.00) & (df["vig"] <= 1.20) &
    (df["over_odds"].abs() >= 5) &
    (df["under_odds"].abs() >= 5)
].copy()

n_clean = len(df_clean)
n_total = len(df)
print(f"Rows passing filter: {n_clean:,} / {n_total:,} ({n_clean/n_total*100:.1f}%)")
print(f"Dropped: {n_total - n_clean:,}")
print(f"  - vig out of range [1.00, 1.20]: {((df['vig'] < 1.00) | (df['vig'] > 1.20)).sum():,}")
print(f"  - near-zero odds (|odds| < 5):   {((df['over_odds'].abs() < 5) | (df['under_odds'].abs() < 5)).sum():,}")

In [ ]:
df_clean["y_under"] = (df_clean["AST"] < df_clean["prop_line"]).astype(int)
df_clean["pnl"] = df_clean.apply(
    lambda r: american_profit(r["under_odds"]) if r["y_under"] == 1 else -1.0, axis=1
)

print("=== Overall ===")
print(f"n={len(df_clean):,}  hit_rate={df_clean['y_under'].mean():.3f}  ROI={df_clean['pnl'].mean()*100:.2f}%")
print()
print("=== By season ===")
print(df_clean.groupby("season")[["y_under","pnl"]].mean().rename(
    columns={"y_under":"hit_rate","pnl":"roi"}).round(3))

In [ ]:
print("=== ROI by line tier ===")
print(df_clean.groupby("line_tier", observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")
).round(3))
print()
print("=== ROI by line tier x season ===")
print(df_clean.groupby(["season", "line_tier"], observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")
).round(3))

In [ ]:
df_clean["p_over_raw"] = df_clean["over_odds"].apply(american_to_implied_prob)
df_clean["p_mkt_dv"] = df_clean["p_over_raw"] / df_clean["vig"]
df_clean["y_over"] = (df_clean["AST"] >= df_clean["prop_line"]).astype(int)

cal = df_clean.groupby("line_tier", observed=True).apply(lambda g: pd.Series({
    "mkt_p_over": g["p_mkt_dv"].mean(),
    "actual_over": g["y_over"].mean(),
    "gap": g["p_mkt_dv"].mean() - g["y_over"].mean(),
    "n": len(g),
}), include_groups=False)
print("=== Calibration gap by line tier ===")
print("(gap > 0 means market overprices over / underprices under)")
print(cal.round(4))

In [ ]:
best_tier_label = "mid (2.5-3.5)"
best_tier = df_clean[df_clean["line_tier"] == best_tier_label].copy()

best_tier["min_bucket"] = pd.cut(best_tier["MIN"], bins=[10, 20, 28, 36, 100],
                                  labels=["10-20","20-28","28-36","36+"])
print(f"=== Minutes segmentation: {best_tier_label} ===")
print(best_tier.groupby("min_bucket", observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")).round(3))
print()
print(f"=== By season: {best_tier_label} ===")
print(best_tier.groupby("season")[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")).round(3))

In [ ]:
dc = df_clean.sort_values(["PLAYER_NAME","season","game_date"]).copy()
dc["ast_roll10"] = dc.groupby(["PLAYER_NAME","season"])["AST"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=5).mean()
)
dc["line_vs_roll"] = dc["prop_line"] - dc["ast_roll10"]

print("=== Under ROI by (prop_line - ast_roll10) bucket ===")
dc["lag_bucket"] = pd.cut(dc["line_vs_roll"],
    bins=[-20, -2, -1, 0, 1, 2, 20],
    labels=["line << avg", "line < avg", "line ~avg-", "line ~avg+", "line > avg", "line >> avg"]
)
print(dc.groupby("lag_bucket", observed=True)[["y_under","pnl"]].agg(
    count=("pnl","count"), hit_rate=("y_under","mean"), roi=("pnl","mean")).round(3))
print()
print(f"Rows with valid roll10: {dc['ast_roll10'].notna().sum():,} / {len(dc):,}")